In [11]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from transformers import EarlyStoppingCallback
import torch

In [3]:
path = "C:/Users/marts/Downloads/df_ml_good_with_features.csv"
df = pd.read_csv(path)
df_amyloid = df[df["class"] == 1]

print(df_amyloid.columns)

Index(['id', 'sequence', 'length', 'class', 'merge_id', 'a3vSA', 'AAT',
       'Na4vSS', 'THSA', 'nHS', 'TA', 'net_charge', 'hydrophobicity',
       'polar_fraction', 'nonpolar_fraction', 'acidic_fraction',
       'basic_fraction', 'aromatic_fraction', 'beta_propensity', 'seq_entropy',
       'proline_fraction'],
      dtype='object')


In [4]:
dataset = Dataset.from_pandas(df_amyloid[['sequence']])

dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

In [5]:
tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
tokenizer.pad_token = tokenizer.eos_token


def tokenize_function(example):
    tokens = tokenizer(
        example["sequence"],
        truncation=True,
        padding="max_length",
        max_length=50
    )
    labels = tokens["input_ids"].copy()

    #ignorowanie paddingu w loss
    labels = [
        -100 if token == tokenizer.pad_token_id else token
        for token in labels
    ]

    tokens["labels"] = labels
    return tokens


train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/905 [00:00<?, ? examples/s]

Map:   0%|          | 0/101 [00:00<?, ? examples/s]

In [6]:
for i in range(5):
    print(train_dataset[i])

{'sequence': 'QARDSGEIYAAAGDMHI', '__index_level_0__': 380, 'input_ids': [1156, 36, 784, 399, 1888, 594, 553, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [1156, 36, 784, 399, 1888, 594, 553, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}
{'sequence': 'RQVLIF', '__index_level_0__': 807, 'input_ids': [403, 8618, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [403, 8618, 0,

In [7]:
model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2")

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
training_args = TrainingArguments(
    output_dir="files/models/protGPT2-results",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=20,
    do_eval=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=50,
    fp16=True, #gpu
    weight_decay=0.01, #L2
)

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [10]:
trainer.train()

C:\Users\marts\anaconda3\envs\protein310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,0.818997,0.776958


Epoch,Training Loss,Validation Loss
1,0.818997,0.776958
2,0.633425,0.761494


C:\Users\marts\anaconda3\envs\protein310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.818997,0.776958
2,0.633425,0.761494
3,0.569701,0.782158


...

Epoch,Training Loss,Validation Loss
1,0.818997,0.776958
2,0.633425,0.761494
3,0.569701,0.782158
4,0.451969,0.867085
5,0.338494,0.966525


C:\Users\marts\anaconda3\envs\protein310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.818997,0.776958
2,0.633425,0.761494
3,0.569701,0.782158
4,0.451969,0.867085
5,0.338494,0.966525


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=2265, training_loss=0.5827522322042099, metrics={'train_runtime': 16252.8243, 'train_samples_per_second': 1.114, 'train_steps_per_second': 0.557, 'total_flos': 961639968000000.0, 'train_loss': 0.5827522322042099, 'epoch': 5.0})

In [23]:
model_path = "files/models/protGPT2-results/checkpoint-906"

tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
model = AutoModelForCausalLM.from_pretrained(model_path)

model.eval()

prompt = "G"

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=50,
        min_length=6,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        num_return_sequences=10
    )

sequences = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

for i, seq in enumerate(sequences):
    print(f"\nSequence {i+1}:")
    print(seq)

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.



Sequence 1:
GHIYQNNSVSGGTQHAPVVQGHTF

Sequence 2:
GNIVIGDNA


Sequence 3:
GVAVRGDGAVMVGNV

Sequence 4:
GNITFTGDA


Sequence 5:
GYQTGPVVQGRDFSG

Sequence 6:
GNIAAGNLTVGNN

Sequence 7:
GTVNLQGAVTGDVVQAGRDISG

Sequence 8:
GQIYQGSI
KGRARVAQG

Sequence 9:
GNVYTQNYTFQ

Sequence 10:
GNVVTAXQ
